In [10]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

In [2]:
df_property = pd.read_parquet('../data/processed/properties_clean.parquet')

df_property

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,tenure,propertyType,currentEnergyRating,log_price
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,Freehold,Bungalow Property,E,13.071070
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,Freehold,Bungalow Property,C,12.971540
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,Freehold,Bungalow Property,D,13.142166
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,Freehold,Bungalow Property,E,12.936034
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,Freehold,Bungalow Property,D,13.972514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10661,"Flat 2B, Avenue Studios, Sydney Close, London,...",SW3 6HW,1750000,2024-04-05,51.491681,-0.172813,1.0,1.0,1.0,120.0,Leasehold,Terraced,E,14.375126
10662,"99 Victorian Grove, London, N16 8EJ",N16 8EJ,580000,2024-09-13,51.557519,-0.076808,2.0,1.0,1.0,89.0,Leasehold,Terraced,D,13.270783
10663,"9A Cambray Road, London, SW12 0DX",SW12 0DX,266092,2024-07-24,51.444387,-0.140667,2.0,2.0,1.0,78.0,Leasehold,Terraced,C,12.491597
10664,"20 Barfett Street, London, W10 4NP",W10 4NP,715000,2024-09-04,51.527446,-0.206082,2.0,1.0,1.0,67.0,Freehold,Terraced Bungalow,D,13.480038


In [3]:
df_property['total_rooms'] = df_property['bedrooms'] + df_property['bathrooms'] + df_property['livingRooms']

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,tenure,propertyType,currentEnergyRating,log_price,total_rooms
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,Freehold,Bungalow Property,E,13.071070,3.0
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,Freehold,Bungalow Property,C,12.971540,3.0
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,Freehold,Bungalow Property,D,13.142166,4.0
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,Freehold,Bungalow Property,E,12.936034,4.0
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,Freehold,Bungalow Property,D,13.972514,4.0


In [4]:
df_property['bath_bed_ratio'] = df_property['bathrooms'] / df_property['bedrooms'].replace(0, 1)

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,tenure,propertyType,currentEnergyRating,log_price,total_rooms,bath_bed_ratio
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,Freehold,Bungalow Property,E,13.071070,3.0,1.0
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,Freehold,Bungalow Property,C,12.971540,3.0,1.0
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,Freehold,Bungalow Property,D,13.142166,4.0,0.5
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,Freehold,Bungalow Property,E,12.936034,4.0,0.5
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,Freehold,Bungalow Property,D,13.972514,4.0,0.5


In [5]:
df_property['area_per_room_sqm'] = df_property['floorAreaSqM'] / df_property['total_rooms']

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,tenure,propertyType,currentEnergyRating,log_price,total_rooms,bath_bed_ratio,area_per_room_sqm
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,Freehold,Bungalow Property,E,13.071070,3.0,1.0,12.333333
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,Freehold,Bungalow Property,C,12.971540,3.0,1.0,20.333333
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,Freehold,Bungalow Property,D,13.142166,4.0,0.5,16.250000
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,Freehold,Bungalow Property,E,12.936034,4.0,0.5,19.250000
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,Freehold,Bungalow Property,D,13.972514,4.0,0.5,20.750000


In [6]:
df_property['is_freehold'] = (df_property['tenure'] == 'Freehold').astype(int)

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,tenure,propertyType,currentEnergyRating,log_price,total_rooms,bath_bed_ratio,area_per_room_sqm,is_freehold
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,Freehold,Bungalow Property,E,13.071070,3.0,1.0,12.333333,1
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,Freehold,Bungalow Property,C,12.971540,3.0,1.0,20.333333,1
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,Freehold,Bungalow Property,D,13.142166,4.0,0.5,16.250000,1
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,Freehold,Bungalow Property,E,12.936034,4.0,0.5,19.250000,1
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,Freehold,Bungalow Property,D,13.972514,4.0,0.5,20.750000,1


In [7]:
energy_map = {
    'A': 7,
    'B': 6,
    'C': 5,
    'D': 4,
    'E': 3,
    'F': 2,
    'G': 1
}

df_property['energy_score'] = df_property['currentEnergyRating'].map(energy_map)

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,tenure,propertyType,currentEnergyRating,log_price,total_rooms,bath_bed_ratio,area_per_room_sqm,is_freehold,energy_score
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,Freehold,Bungalow Property,E,13.071070,3.0,1.0,12.333333,1,3
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,Freehold,Bungalow Property,C,12.971540,3.0,1.0,20.333333,1,5
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,Freehold,Bungalow Property,D,13.142166,4.0,0.5,16.250000,1,4
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,Freehold,Bungalow Property,E,12.936034,4.0,0.5,19.250000,1,3
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,Freehold,Bungalow Property,D,13.972514,4.0,0.5,20.750000,1,4


In [8]:
CENTRE_LAT = 51.50734
CENTRE_LONG = -0.12765
EARTH_RADIUS = 6371

def haversine_distance(lat1, long1, lat2, long2, radius):
    lat1, long1, lat2, long2 = map(np.radians, [lat1, long1, lat2, long2]) # Converting degrees into radians

    lat_distance = lat2 - lat1
    long_distance = long2 - long1

    angular_separation = np.sin(lat_distance / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(long_distance / 2)**2 # Calculating the haversine formula component measuring the angular separation between two points on a spherical surface

    central_angle = 2 * np.arcsin(np.sqrt(angular_separation)) # Calculating the central angle between the two points

    return radius * central_angle

In [9]:
df_property['distance_to_centre_km'] = haversine_distance(df_property['latitude'], df_property['longitude'], CENTRE_LAT, CENTRE_LONG, EARTH_RADIUS)

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,tenure,propertyType,currentEnergyRating,log_price,total_rooms,bath_bed_ratio,area_per_room_sqm,is_freehold,energy_score,distance_to_centre_km
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,Freehold,Bungalow Property,E,13.071070,3.0,1.0,12.333333,1,3,4.172360
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,Freehold,Bungalow Property,C,12.971540,3.0,1.0,20.333333,1,5,6.992492
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,Freehold,Bungalow Property,D,13.142166,4.0,0.5,16.250000,1,4,8.344587
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,Freehold,Bungalow Property,E,12.936034,4.0,0.5,19.250000,1,3,11.435198
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,Freehold,Bungalow Property,D,13.972514,4.0,0.5,20.750000,1,4,6.087600


In [11]:
df_stops = pd.read_parquet('../data/processed/stops_clean.parquet')

df_stops

,atcocode,commonname,latitude,longitude,stoptype,town
0,0170ZZAVBIT0,Bitton (Avon Valley Railway),51.430350,-2.476210,TMU,Unknown
1,0170ZZAVOLD0,Oldland (Avon Valley Railway),51.443660,-2.467350,TMU,Unknown
2,0400ZZLUAMS,Amersham Underground Station,51.674198,-0.607434,TMU,Unknown
3,0400ZZLUAMS0,Amersham Underground Station,51.674206,-0.607362,TMU,Unknown
4,0400ZZLUCAL,Chalfont & Latimer Underground Station,51.668075,-0.560495,TMU,Unknown
...,...,...,...,...,...,...
5161,9400ZZTWWJS,Balfour Street (Edinburgh Trams),55.965364,-3.176258,MET,Unknown
5162,9400ZZTWWJT,McDonald Road (Edinburgh Trams),55.960678,-3.181465,MET,Unknown
5163,9400ZZTWWJU,Picardy Place (Edinburgh Trams),55.956817,-3.186794,MET,Unknown
5164,9400ZZBPTAL,Talbot Road (Blackpool Tramway),53.819144,-3.054507,MET,Unknown


In [ ]:
property_coordinates = np.radians(df_property[['latitude', 'longitude']].values)
stop_coordinates = np.radians(df_stops[['latitude', 'longitude']].values)